# 产业链白名单 v4 — Colab 环境安装
---
跑这个 notebook 之前，先把以下两个文件夹上传到 Google Drive 的 `MyDrive/WhiteList/` 下面：

| 上传内容 | Google Drive 路径 |
|---------|------------------|
| `model_v4/` 所有 .py | `MyDrive/WhiteList/model_v4/` |
| `csmar/` 所有数据 | `MyDrive/WhiteList/csmar/` |

上传完之后依次执行下面每个 Cell 就行。

In [ ]:
# ============================================================
# Cell 1: 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/WhiteList'
print('Drive 挂载成功' if os.path.exists(DRIVE_ROOT) else '请检查路径: ' + DRIVE_ROOT)

In [ ]:
# ============================================================
# Cell 2: 检查 Colab 自带环境
# ============================================================
import sys, subprocess

print('Python:', sys.version.split()[0])

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

print(f'Python 路径: {sys.executable}')

In [ ]:
# ============================================================
# Cell 3: 安装 PyTorch Geometric + 依赖
# ============================================================
# PyG 2.5+ 可以直接 pip install，会自动匹配 torch 版本
!pip install -q torch-geometric

# 辅助库
!pip install -q scikit-learn pandas numpy

print('安装完成，验证版本...')
import torch_geometric
print(f'PyTorch Geometric: {torch_geometric.__version__}')
import sklearn
print(f'scikit-learn: {sklearn.__version__}')

In [ ]:
# ============================================================
# Cell 4: 添加代码路径
# ============================================================
CODE_DIR = os.path.join(DRIVE_ROOT, 'model_v4')
DATA_DIR = os.path.join(DRIVE_ROOT, 'csmar')

print(f'代码目录: {CODE_DIR}')
print(f'数据目录: {DATA_DIR}')
print(f'代码存在: {os.path.exists(CODE_DIR)}')
print(f'数据存在: {os.path.exists(DATA_DIR)}')

# 把 model_v4 加到 Python path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
if os.path.join(CODE_DIR, 'data_loader') not in sys.path:
    sys.path.insert(0, os.path.join(CODE_DIR, 'data_loader'))

In [ ]:
# ============================================================
# Cell 5: 快速导入测试（不加载数据）
# ============================================================
from config import DEVICE, HIDDEN_DIM
from data_interface import HeteroGraphData
from layer2_encoder import DualOutputEncoder
from layer3_gamma import CrossRelationPropagation
from layer4_temporal import TemporalEncoder
from classifier import MultiTaskHeads
from train import RiskWhiteListModel

print('全部模块导入成功')
print(f'config.DEVICE = {DEVICE}')

In [ ]:
# ============================================================
# Cell 6: 修正 CSMAR 数据路径 + 确认数据可访问
# ============================================================
import data_loader.csmar_loader as csmar
csmar.CSMAR_DIR = DATA_DIR

# 检查几个关键 CSV 是否存在
key_files = [
    '偿债能力164658534/FI_T1.csv',
    '盈利能力164846481/FI_T5.csv',
    '十大股东文件170142830/HLD_Shareholders.csv',
]
all_ok = True
for f in key_files:
    p = os.path.join(DATA_DIR, f)
    ok = os.path.exists(p)
    print(f'  {"✓" if ok else "✕"} {f}')
    if not ok:
        all_ok = False

if all_ok:
    print('\n数据文件就绪，可以开始训练')
else:
    print('\n部分数据缺失，请检查 Google Drive 上传是否完整')

---
### 以上全部跑通之后，执行下面这个 Cell 开始训练
如果还没准备好跑全量，先不要执行。

In [ ]:
# ============================================================
# Cell 7: 全量训练（确认环境 OK 后再跑）
# ============================================================
import time
import torch
import numpy as np
import pandas as pd

from train import RiskWhiteListModel, train, evaluate_model
from config import DEVICE, SEED, OUTPUT_DIR

torch.manual_seed(SEED)
np.random.seed(SEED)

print('=' * 60)
print('  产业链白名单 v4 — Colab GPU 训练')
print('=' * 60)

# Load
print('\n[加载数据]...')
data = csmar.load_csmar_data()

# 检查并修正 DEVICE
actual_device = 'cuda' if torch.cuda.is_available() else 'cpu'
if actual_device == 'cuda':
    # 覆盖 config 里的 DEVICE
    import config
    config.DEVICE = 'cuda'

# Model
print(f'\n[初始化] 模型 v4 (device={actual_device})...')
in_dims = {nt: feat.shape[1] for nt, feat in data.x_dict.items()}
model = RiskWhiteListModel(
    in_dims=in_dims,
    edge_types=list(data.edge_index_dict.keys())
)
print(f'  参数量: {sum(p.numel() for p in model.parameters()):,}')

# Train
model = train(model, data)

# Eval
test_auc, test_acc, gamma = evaluate_model(model, data, data.test_mask)
print(f'\n测试 AUC: {test_auc:.4f}  |  测试 Acc: {test_acc:.4f}')

# Gamma
edge_names = model.edge_names
print(f'\nΓ 矩阵:')
header = '  ' + '  '.join(f'{e:>10s}' for e in edge_names)
print(header)
gamma_np = gamma.cpu().detach().numpy()
for i, ename_i in enumerate(edge_names):
    vals = '  '.join(f'{gamma_np[i,j]:10.3f}' for j in range(len(edge_names)))
    print(f'  {ename_i:>10s}: {vals}')

print(f'\n完成')